# Walkthrough

This notebook goes through the analysis step by step, using the same code that
`scripts/run_all.py` runs. It is meant for reading: each section explains one
idea and shows the numbers behind it.

You need the data files in `data/` (see `data/README.md`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from rct import tables
from rct.data import load_study
from rct.metrics import confusion_counts, metrics_from_counts, participant_metrics, to_itt, drop_outliers

pd.set_option("display.width", 160)
study = load_study(ROOT / "data")
print(len(study.all_ids), "sentences,", len(study.reference), "reference errors in", len(study.gold), "sentences")
print("Randomized:", study.n_randomized)

585 sentences, 142 reference errors in 100 sentences
Randomized: {'Control': 40, 'Experimental': 38}


## 1. The reference standard

The reference standard lists 142 errors. They sit in 100 of the 585 sentences, so
the analysis is at sentence level: a sentence either has an error or it does not.

In [2]:
tables.table2_reference_standard(study)

,Characteristic,n,%
0,Excerpt,<NA>,NaN
1,Pages,20,NaN
2,Sentences,585,100.0
3,Sentences with at least one error,100,17.1
4,Error-free sentences,485,82.9
5,Errors (total = 142),<NA>,NaN
6,Orthographic,74,52.1
7,Grammatical,68,47.9
8,Orthographic subtypes,<NA>,NaN
9,Punctuation,38,51.4


## 2. What counts as a flagged sentence

- **Control:** every record that could be matched to a sentence.
- **Experimental:** the educator's own records, plus LLM suggestions they accepted.
  A suggestion they rejected counts as not flagged.

The flagged sentences of one participant are `study.positives[(arm, participant)]`.

In [3]:
arm, participant = "Experimental", study.exposure.query("arm == 'Experimental'").participant.iloc[0]
print(participant, "flagged sentences:", sorted(study.positives.get((arm, participant), set())))
print("Sentences the LLM flagged (whatever the educators decided):", len(study.llm_flagged))

P001 flagged sentences: []
Sentences the LLM flagged (whatever the educators decided): 38


## 3. Exposure

The task was time limited, so many participants never reached the last pages. A
participant's exposure is every sentence up to the last page where they registered
a record. Recall, F1, accuracy and kappa are computed inside that window only.
Precision does not depend on it, because every flagged sentence is already inside it.

Participants with no record at all have no way of telling how far they read.

In [4]:
study.exposure.groupby("arm").agg(participants=("participant", "size"),
                                  median_last_page=("last_page", "median"),
                                  max_last_page=("last_page", "max"))

,participants,median_last_page,max_last_page
arm,,,
Control,26,143.0,155
Experimental,37,145.0,160


In [5]:
per_protocol = participant_metrics(study)      # participants with at least one record
itt = to_itt(per_protocol, study.n_randomized) # + participants without records, scored 0 (40 / 38)
per_protocol.head()

,Arm,Participant,LastPage,TP,FP,FN,TN,Responses,Correct,Precision,Recall,F1,Accuracy,Kappa
0,Control,P002,141,0,0,3,28,0,0,0.000000,0.000000,0.000000,0.903226,0.000000
1,Control,P009,141,0,1,3,27,1,0,0.000000,0.000000,0.000000,0.870968,-0.050847
2,Control,P013,143,0,1,12,80,1,0,0.000000,0.000000,0.000000,0.860215,-0.020253
3,Control,P014,146,3,2,23,137,5,3,0.600000,0.115385,0.193548,0.848485,0.150360
4,Control,P015,155,5,7,72,361,12,5,0.416667,0.064935,0.112360,0.822472,0.068914


## 4. Metrics, step by step

For one participant: build the confusion counts inside their window and compute the metrics.

In [6]:
row = per_protocol.iloc[0]
window = study.universe_upto(int(row.LastPage))
counts = confusion_counts(study.positives.get((row.Arm, row.Participant), set()), study.gold, window)
print(row.Participant, "| sentences seen:", len(window), "| TP, FP, FN, TN =", counts)
{k: round(v, 3) for k, v in metrics_from_counts(*counts).items() if k in ["Precision", "Recall", "F1", "Accuracy", "Kappa"]}

P002 | sentences seen: 31 | TP, FP, FN, TN = (0, 0, 3, 28)


{'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'Accuracy': 0.903, 'Kappa': 0.0}

## 5. Control vs experimental (Tables 3 and 4)

Mann-Whitney U tests (two-sided). In Table 4 the p-values are adjusted with Holm over the four metrics.

In [7]:
descriptives, tests = tables.table3_descriptives(itt)
display(descriptives)
display(tests)

,Control - Total Responses,Control - Precision,Experimental - Total Responses,Experimental - Precision
Mean,1.88,36.52%,5.42,86.83%
SD,2.65,42.78%,7.49,24.84%
Minimum,0.00,0.00%,0.00,0.00%
1st Quartile (25%),0.00,0.00%,1.25,81.88%
Median (50%),1.00,0.00%,3.00,100.00%
3rd Quartile (75%),3.00,81.88%,6.00,100.00%
Maximum,12.00,100.00%,35.00,100.00%


,Variable,U,p
0,Responses,387.0,0.0002
1,Precision,300.0,<0.0001


In [8]:
tables.table4_quality(itt)

,Metric,Control,Experimental,U,p-value (Holm)
0,Mean Accuracy (SE),56.76% (6.68),84.77% (2.36),558.0,0.0430
1,Mean Recall (SE),9.65% (2.63),17.81% (1.66),344.0,<0.0001
2,Mean F1-Score (SE),13.65% (3.26),28.33% (2.31),337.5,<0.0001
3,Mean Kappa (SE),0.1154 (0.0313),0.2512 (0.0222),324.0,<0.0001


## 6. AI-only vs AI-assisted (Table 5)

*AI-only* is every sentence the LLM flagged. *AI-assisted* pools the sentences flagged
by all experimental educators (the union), so it measures the group, not one person.

Because it is a union, a participant who flagged nothing cannot change it. That is why the
pooled rows for 37 educators (with records) and 38 (all randomized) are identical.

In [9]:
main, top5, per_exposure, mcnemar = tables.table5_ai_vs_assisted(study)
main

,Condition,Accuracy,Precision,Recall,F1-Score,Cohen's Kappa
0,AI-Only,0.8496,0.6579,0.25,0.3623,0.2960
1,"AI-Assisted (pooled, n=37)",0.8513,0.6327,0.31,0.4161,0.3421
2,"AI-Assisted (pooled, n=38)",0.8513,0.6327,0.31,0.4161,0.3421


Individual educators, ranked by their own F1-score. Each one is compared with the
LLM alone with an exact McNemar test at sentence level.

In [10]:
top5

,Condition,Accuracy,Precision,Recall,F1-Score,Cohen's Kappa
0,AI-Only,0.8496,0.6579,0.25,0.3623,0.2960
1,"AI-Assisted (pooled, n=37)",0.8513,0.6327,0.31,0.4161,0.3421
2,Educator 1,0.8513,0.6970,0.23,0.3459,0.2852
3,Educator 2,0.8479,0.6571,0.23,0.3407,0.2766
4,Educator 3,0.8479,0.9231,0.12,0.2124,0.1801
5,Educator 4,0.8427,1.0000,0.08,0.1481,0.1260
6,Educator 5,0.8410,0.8889,0.08,0.1468,0.1220


In [11]:
mcnemar

,Comparison with AI-Only,LLM only,Educators only,McNemar p
0,AI-Assisted (pooled),0,11,0.0010
1,Educator 1,5,0,0.0625
2,Educator 2,3,0,0.2500
3,Educator 3,25,0,<0.0001
4,Educator 4,32,2,<0.0001
5,Educator 5,29,0,<0.0001


A different way to bring every educator in is to sum each educator's confusion counts
inside the pages that educator reached. Recall is lower here because it is measured
per educator, not on the union.

In [12]:
per_exposure

,Condition,Accuracy,Precision,Recall,F1-Score,Cohen's Kappa
0,"AI-Assisted (per-educator exposure, n=37)",0.8538,0.835,0.1552,0.2618,0.221


## 7. Sensitivity (Table 7)

Group means for four samples: all randomized participants or only those with records,
each with and without outliers. Outliers are found with Tukey's fences (1.5 x IQR) on total
responses, separately inside each group.

In [13]:
tables.table7_sensitivity(per_protocol, itt)

,Sample,Precision,Recall,F1,Kappa,"p (Holm, range)"
0,All randomized (40/38),36.52 / 86.83,9.65 / 17.81,13.65 / 28.33,0.115 / 0.251,<0.0001 - <0.0001
1,"All, no outliers (37/36)",33.96 / 87.90,7.64 / 17.52,11.32 / 28.00,0.094 / 0.250,<0.0001 - <0.0001
2,>=1 record (26/37),56.19 / 89.18,14.85 / 18.29,21.00 / 29.10,0.178 / 0.258,0.0042 - 0.0239
3,">=1 record, no outliers (23/34)",54.64 / 90.35,12.28 / 17.95,18.22 / 28.65,0.152 / 0.256,0.0065 - 0.0107


In [14]:
_, removed = drop_outliers(itt)
removed[["Arm", "Participant", "Responses", "Correct", "LastPage"]]

,Arm,Participant,Responses,Correct,LastPage
4,Control,P015,12.0,5.0,155.0
15,Control,P048,8.0,7.0,145.0
19,Control,P057,8.0,6.0,142.0
41,Experimental,P036,33.0,23.0,160.0
53,Experimental,P065,35.0,23.0,160.0


## 8. Participation

14 of 40 control participants and 1 of 38 experimental participants registered nothing.

In [15]:
tables.participation(study, per_protocol)

,Arm,Randomized,No records,% no records,Median pages reached (with records),Fisher exact p
0,Control,40,14,35.0,3.0,0.0003
1,Experimental,38,1,2.6,5.0,
